# Zerlaut parameter exploration

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import json5 as json
from concurrent.futures import ProcessPoolExecutor, as_completed
from contextlib import redirect_stdout, redirect_stderr
from copy import deepcopy
from time import perf_counter
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
from io import StringIO

import logging

# from joblib import Parallel, delayed


In [ ]:
from aux.environment import resolve_directory
from aux.parameters import bounds_from_default, filter_nested_dict, add_beta_config_safe
from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
from aux.tvb.plotting import plot_results
from aux.parameters import flatten_params, restore_flat_params
from aux.sampling import sample_params
from aux.tvb.stimulus import build_stimulus
from aux.tvb.model import build_model, get_model_class
from aux.tvb.integrator import build_integrator
from aux.tvb.simulator import build_single_region_simulator


**Console**

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print
pprint(f'The console is "ready" to print messages with rich formatting.')

**Set main directory paths**

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Auxiliar Functions

In [ ]:
def remove_unsampled_parameters(params, names) -> dict:
    result = deepcopy(params)
    for key in names:
        result.pop(key, None) # Evita el error si no se encuentra
    return result

In [ ]:
def count_parameters(params):
    return sum(
        len(v['default']) if isinstance(v['default'], list) else 1 
        for v in params.values()
    )

## Configuration

### Global configuration

In [ ]:
CONFIG_DIR = resolve_directory(settings['CONFIG_DIR'], env_dir) / "zerlaut"
CONFIG_FILE_NAME = "test_config.json"

pprint(f'Configuration directory: {CONFIG_DIR}')

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
BATCH_SIZE = 2**5 #32
SEED = 42

In [ ]:
OUTPUT_DTYPE = "float32"
# OUTPUT_STATES = None # Guarda las variables por defecto del modelo.
OUTPUT_STATES = "all" # Guarda todas las variables del modelo.
# OUTPUT_STATES = ['E', 'I', 'W_e', 'W_i', 'ou_drift']

In [ ]:
DURATION_MS = 8000.0    # Duración de la simulación en milisegundos.
DT_MS = 0.2             # Paso de integración en milisegundos.
ONSET_MS = 500.0        # Tiempo de inicio del estímulo en milisegundos.

INTEGRATOR_NAME = 'HeunDeterministic' # Esto permite quitar dos parametros de ruido, puedes meter ruido aditivo en el estimulo
#INTEGRATOR_NAME = 'HeunStochastic'

MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"
#INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}

PARAMETER_CONFIG = "CONFIG2" 
STIMULUS_CONFIG = "CONFIG2"
STIMULUS_NOISE_ON = False # Meter ruido aditivo, tal vez sea ridiculo teniendo el propio modelo de ruido del modelo.

In [ ]:
# from tvb.simulator.models import ModelsEnum
# from tvb.simulator import integrators as Integrators

# pprint(f"Modelos disponibles: {sorted(model.value for model in ModelsEnum)}")

# integrator_names = sorted(
#     name for name, cls in vars(Integrators).items()if isinstance(cls, type) and issubclass(cls, Integrators.Integrator)
# )

# pprint(f"Integradores disponibles: {integrator_names}") # HeunDeterministic, HeunStochastic

### Parameter sampling


In [ ]:
CONFIG_PATH = WORK_DIR / 'info' / 'zerlaut' / 'params_raw.json'
pprint(f'Parameter configuration: {CONFIG_PATH}')

In [ ]:
with CONFIG_PATH.open(encoding='utf-8') as config_file:
    PARAMS = json.load(config_file)

SAMPLED_PARAMS, FIXED_PARAMS = deepcopy(PARAMS), {}
SAMPLED_PARAMS = filter_nested_dict(SAMPLED_PARAMS, ['min', 'max', 'default'])

pprint(f'Loaded {count_parameters(PARAMS)}', str(list(PARAMS.keys())))

#### Common processing

1. **Remove unsampled parameters**

In [ ]:
SAMPLED_PARAMS = remove_unsampled_parameters(SAMPLED_PARAMS, [
    "external_input_ex_ex", "external_input_ex_in", "external_input_in_ex", "external_input_in_in" #,S_i
])

2. **Add undefined bounds**

In [ ]:
SAMPLED_PARAMS = bounds_from_default(SAMPLED_PARAMS, names=["P_e", "P_i"], margin=0.10)

for param in (SAMPLED_PARAMS['P_e'], SAMPLED_PARAMS['P_i']):
    p = param['default'][0]
    param['min'][0], param['max'][0] = p - 0.05 * abs(p), p + 0.05 * abs(p)

# NOTE: Puede merecer la pena usar una distribución beta para el muetreo

3. **Adjust noise**

In [ ]:
## Reduce noise

SAMPLED_PARAMS['tau_OU'] |= {'min': 0.05, 'max': 4, 'default': 2}
SAMPLED_PARAMS['weight_noise'] |= {'min': 0.00001, 'max': 0.0004, 'default': 0.0001}

if 'Stochastic' not in INTEGRATOR_NAME:
    remove_unsampled_parameters(SAMPLED_PARAMS, ['tau_OU', 'weight_noise'])

# NOTE: Puede merecer la pena usar una distribución beta para el muetreo

4. **Cell**

In [ ]:
# SAMPLED_PARAMS['g_L'] |= {'min': 1.0, 'max': 50.0, 'default': 10.0}
# SAMPLED_PARAMS['C_m'] |= {'min': 50, 'max': 250, 'default': 200}


5. **Adaptation**

TODO;

Son inestables si crecen los 2!

Exploring parameters: ['E_e', 'E_i']
Default: [0.0, -80.0]
Min: [-20.0, -100.0]
Max: [20.0, -60.0]

Insestables para valores altos de Q_e con valores bajos de Q_i

Exploring parameters: ['E_e', 'E_i']
Default: [0.0, -80.0]
Min: [-20.0, -100.0]
Max: [20.0, -60.0]

No dejar subir Q_e de dos!

In [ ]:
# SAMPLED_PARAMS["Q_e"] |= {'min': 0.5, 'max': 2.0, 'default': 1.0}

5. $S_i$

In [ ]:
SAMPLED_PARAMS['S_i'] |= {'min': 0, 'max': 1.3, 'default': 1}

#### Parameter configuration 1

* Add beta distiribution to all

In [ ]:
if PARAMETER_CONFIG in ("CONFIG1", "CONFIG2"):
    SAMPLED_PARAMS = add_beta_config_safe(SAMPLED_PARAMS, lambda_=2)

#### Parameter configuration 2

* Fix $S_i$ to 0.

In [ ]:
if PARAMETER_CONFIG== "CONFIG2":
    FIXED_PARAMS["S_i"] = 0
    CONFIG_FILE_NAME = "test_config2.json"

### Stimulus configuration


In [ ]:
ex_ex_inputs_hfreq, ex_ex_inputs_hfreq_probs = [{}], [1]
ex_ex_inputs_lfreq, ex_ex_inputs_lfreq_probs = [{}], [1]
ex_ex_inputs_noise, ex_ex_inputs_noise_probs = [{}], [1]
ex_in_inputs_hfreq, ex_in_inputs_hfreq_probs = [{}], [1]
in_ex_inputs, in_ex_inputs_probs = [{}], [1]


In [ ]:
if STIMULUS_CONFIG in ("CONFIG1", "CONFIG2"):
    ex_ex_inputs_hfreq = [
        {"random_smooth": [{
            "baseline_khz": 0.001, "onset_ms": ONSET_MS,
            "min_duration_ms": 100.0, "max_duration_ms": 250.0,
            "min_amplitude_khz": 0.001, "max_amplitude_khz": 0.01,
        }]},
        {"random_smooth": [{
            "baseline_khz": 0.005, "onset_ms": ONSET_MS,
            "min_duration_ms": 100.0, "max_duration_ms": 250.0,
            "min_amplitude_khz": 0.005, "max_amplitude_khz": 0.07,
        }]},
    ]

    ex_ex_inputs_hfreq_probs = [0.5, 0.5]

    ex_ex_inputs_lfreq = [{},
        {"random_smooth": [{
            "baseline_khz": 0, "onset_ms": ONSET_MS,
            "min_duration_ms": 30.0, "max_duration_ms": 70.0,
            "min_amplitude_khz": 0.0, "max_amplitude_khz": 0.005,
        }]}, 
    ]

    ex_ex_inputs_lfreq_probs = [0.5, 0.5]

    # Solo se mete ruido par los deterministas.
    ex_ex_inputs_noise = [{},
            {"random_std_noise": {"mean_khz": 0.0, "min_std_khz": 0.0001, "max_std_khz": 0.0005}}
    ]

    ex_ex_inputs_noise_probs = [0.5, 0.5]

    in_ex_inputs = [{"external_input_ex_ex": 1}] if STIMULUS_CONFIG == "CONFIG1" else [{}]
    in_ex_inputs_probs = [1.0]

In [ ]:
stimulus_configs, stimulus_probs = [], []
 
if 'Stochastic' not in INTEGRATOR_NAME or not STIMULUS_NOISE_ON:
    ex_ex_inputs_noise, ex_ex_inputs_noise_probs = [{}], [1.0]

for (hfreq, p_hfreq), (lfreq, p_lfreq), (noise, p_noise), (in_ex, p_in_ex) in product(
    zip(ex_ex_inputs_hfreq, ex_ex_inputs_hfreq_probs),
    zip(ex_ex_inputs_lfreq, ex_ex_inputs_lfreq_probs),
    zip(ex_ex_inputs_noise, ex_ex_inputs_noise_probs),
    zip(in_ex_inputs, in_ex_inputs_probs),
):
    # Creación de ex_ex
    ex_ex = deepcopy(hfreq) 
    for kind, signals in lfreq.items():
        ex_ex.setdefault(kind, []).extend(deepcopy(signals))
    ex_ex.update(deepcopy(noise))

    
    stimulus_configs.append({
        "external_input_ex_ex": ex_ex,
        "external_input_in_ex": deepcopy(in_ex),
    })
    
    stimulus_probs.append(p_hfreq * p_lfreq * p_noise * p_in_ex)


In [ ]:
time_ms = np.arange(DT_MS, DURATION_MS + DT_MS, DT_MS)
n_rows = (len(stimulus_configs) + 1) // 2
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 3 * n_rows), squeeze=False, constrained_layout=True)

for index, (config, probability, ax) in enumerate(
    zip(stimulus_configs, stimulus_probs, axes.flat), start=1
):
    if not any(config.values()):
        ax.set_title(f"No stimulus (p={probability:g})")
        ax.set_axis_off()
        continue
    metadata = ResultMetadata(
        simulation=SimulationMetadata(DURATION_MS, DT_MS),
        model=ModelMetadata(stimulus_units={name: "kHz" for name in config}),
        stimulus=config,
        stimulus_seed=SEED,
    )
    _, stimulus_axes = plot_results(time_ms, {}, metadata, ax=ax)
    stimulus_axes[0].set_title(f"Stimulus {index} (p={probability:g})")

for ax in axes.flat[len(stimulus_configs):]:
    ax.set_visible(False)
plt.show()


#### JSON Building

In [ ]:
# Si vas a dejar un parametro fijo, quitalo.
SAMPLED_PARAMS = remove_unsampled_parameters(SAMPLED_PARAMS, [p for p in FIXED_PARAMS.keys()])

config = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "simulation": {
        "duration_ms": DURATION_MS,
        "dt_ms": DT_MS,
        "integrator": INTEGRATOR_NAME,
    },
    "model": {
        "name": MODEL_NAME,
        "sampled_parameters": deepcopy(SAMPLED_PARAMS),
        "fixed_parameters": deepcopy(FIXED_PARAMS),
    },
    "stimulus": {
        "name": STIMULUS_CONFIG,
        "configurations": deepcopy(stimulus_configs),
        "probabilities": list(stimulus_probs),
    },
    "output": {
        "dtype": OUTPUT_DTYPE,
        "states": deepcopy(OUTPUT_STATES),
    },
}


pprint(f'Loaded {count_parameters(SAMPLED_PARAMS)}', str(list(SAMPLED_PARAMS.keys())))

In [ ]:
with (CONFIG_DIR / CONFIG_FILE_NAME).open("w", encoding="utf-8") as config_file:
    json.dump(config, config_file, indent=2, quote_keys=True, trailing_commas=False)

## Launch sampling

TODO: El sistema de log es mejorrable

In [ ]:
def validate_config(config):
    errors = []
    if config.get("seed", 42) < 0:
        errors.append("seed must be non-negative")
    if config.get("batch_size", 32) <= 0:
        errors.append("batch_size must be positive")
    for name in ("duration_ms", "dt_ms"):
        if config["simulation"][name] <= 0:
            errors.append(f"simulation.{name} must be positive")

    for name, values in config["model"]["sampled_parameters"].items():
        lengths = {len(value) for value in values.values() if isinstance(value, list)}
        if lengths and (len(lengths) != 1 or 0 in lengths
                        or not all(isinstance(value, list) for value in values.values())):
            errors.append(f"{name}: vector fields must have the same length")
            continue
        for key, parameter in flatten_params({name: values}).items():
            minimum, maximum = parameter["min"], parameter["max"]
            if minimum > maximum:
                errors.append(f"{key}: min exceeds max")
            method = parameter.get("method", "uniform")
            if method not in ("uniform", "beta", "truncnorm"):
                errors.append(f"{key}: unknown sampling method '{method}'")
            elif method == "beta":
                if not minimum <= parameter["default"] <= maximum:
                    errors.append(f"{key}: beta default outside [min, max]")
                if parameter.get("lambda", -1) < 0:
                    errors.append(f"{key}: beta lambda must be non-negative")
            elif method == "truncnorm" and parameter.get("std", 0) <= 0:
                errors.append(f"{key}: truncnorm std must be positive")

    configurations = config["stimulus"]["configurations"]
    probabilities = np.asarray(config["stimulus"]["probabilities"])
    if not configurations or len(configurations) != len(probabilities):
        errors.append("stimulus: expected one probability per configuration")
    if np.any(probabilities < 0) or not np.isclose(probabilities.sum(), 1.0, rtol=0, atol=1e-8):
        errors.append("stimulus: probabilities must be non-negative and sum to one")

    if errors:
        raise ValueError("Invalid configuration:\n- " + "\n- ".join(errors))


In [ ]:
INTEGER_PARAMETERS_BY_MODEL = {
    "ZerlautAdaptationFirstOrder": {"N_tot", "K_ext_e", "K_ext_i"},
    "ZerlautAdaptationSecondOrder": {"N_tot", "K_ext_e", "K_ext_i"},
}

NOISE_VARIABLES_BY_MODEL = {
    "ZerlautAdaptationFirstOrder": ["ou_drift"],
    "ZerlautAdaptationSecondOrder": ["ou_drift"],
}

# Parámetros de configuración de ruido para 
def build_noise_config(model_name):
    nsig = [0] * len(state_variables:=get_model_class(model_name).state_variables) 

    for var in NOISE_VARIABLES_BY_MODEL[model_name]:
        nsig[state_variables.index(var)] = 1

    return {"name": "Additive", "params": {"nsig": nsig}}


def simulate_trial(config, parameters, stimulus, n_steps):
    # 3.1 Config
    model_config, simulation_config = config["model"], config["simulation"]
    output_states = (output_config := config["output"])["states"]

    if output_states == "all":
        output_states = list(get_model_class(model_config["name"]).state_variables)

    # 3.2 Creación del modelo
    model = build_model(
        parameters, INTEGER_PARAMETERS_BY_MODEL[model_config["name"]],
        model_class=model_config["name"], variable_of_interest=output_states,
    )

    # 3.3 Creación del estímulo
    integrator = build_integrator(
        simulation_config["dt_ms"], integrator_class=simulation_config["integrator"],
        noise_config=build_noise_config(model_config["name"])
        if "Stochastic" in simulation_config["integrator"] else None,
    )

    # 3.4 Creación del simulador
    simulator = build_single_region_simulator(model, stimulus, integrator)
    time_ms, raw = simulator.run(n_steps=n_steps)[0]

    # 3.5 Resultados. Solo un modo y una región.
    states = {
        name: np.asarray(raw[:, index, 0, 0], dtype=output_config["dtype"])
        for index, name in enumerate(model.variables_of_interest)
    }
    
    return time_ms, states


In [ ]:
def build_trial_stimulus(config, batch_id, trial_id, batch_seq_seed):
    # 2.1. Generación de semillas para la selección y generación de estímulos
    stimulus_selection_seed, stimulus_generation_seed = np.random.SeedSequence(
        batch_seq_seed, spawn_key=(batch_id, trial_id)
    ).spawn(2)

    stimulus_generation_seed = int(
        stimulus_generation_seed.generate_state(1, dtype=np.uint64)[0] 
    ) # debe ser un número para guardar el resultado.

    selection_rng = np.random.default_rng(stimulus_selection_seed) # se usa directamente como semilla

    # 2.2 Selección de un estímulo
    stim_conf = config["stimulus"]
    stimulus_index = int(selection_rng.choice(len(stim_conf["configurations"]), p=stim_conf["probabilities"]))
    stimulus_config = deepcopy(stim_conf["configurations"][stimulus_index])

    # 2.3 Generación del estímulo
    time_ms, stimulus = build_stimulus(
        stimulus_config, config["simulation"]["duration_ms"],
        config["simulation"]["dt_ms"], seed=stimulus_generation_seed,
    )

    # 2.4. Creación de metadatos del estímulo
    stimulus_metadata = {
        "stimulus": stimulus_config,
        "stimulus_seed": stimulus_generation_seed, 
        # "stimulus_probability": float(config["stimulus"]["probabilities"][stimulus_index]),
    }
    return time_ms, stimulus, stimulus_metadata


In [ ]:
def save_config(config, output_dir):
    (output_dir := Path(output_dir)).mkdir(parents=True, exist_ok=True)

    # Comprueba si el fichero existe y si coicide con el de la carpeta de salida. Sino da error.
    file_path = output_dir / "config.json"
    if file_path.exists():
        if json.loads(file_path.read_text(encoding="utf-8")) != config:
            raise ValueError("Output folder configuration does not match config.json")
    else:
        file_path.write_text(json.dumps(config, indent=2, quote_keys=True, trailing_commas=False), encoding="utf-8")


# TODO: es una función distinta a la que usan los cuadernos 01a-01e. Lo ideal sería homogeneizar.  Problema
#     * **Ruido**, unidades
def save_single_region_simulation_(file_path, time_ms, states, metadata):
    file_path = Path(file_path)
    with file_path.with_suffix(".tmp").open("wb") as output:
        np.savez(output, time_ms=time_ms, **states, metadata_json=np.array(json.dumps(metadata)))
    file_path.with_suffix(".tmp").replace(file_path)


def simulate_batch(config, batch_id, sobol_seed, batch_seq_seed, output_dir):
    batch_size = config.get("batch_size", 32)
    sampled_parameters = flatten_params(config["model"]["sampled_parameters"])

    for trail_id, (parameters, probabilities) in enumerate(
        sample_params(
            sampled_parameters, n_samples=batch_size,
            fordware=batch_id * batch_size, seed=sobol_seed,
        )
    ):
        # 1. Parámetros de configuración
        parameters = restore_flat_params(parameters) | deepcopy(config["model"]["fixed_parameters"])
        simulation_id = batch_id * batch_size + trail_id
        file_path = output_dir / f"trial_{simulation_id:08d}.npz"
        if file_path.exists(): continue

        # 2. Generación de los estimulos
        time_ms, stimulus, stimulus_metadata = build_trial_stimulus(
            config, batch_id, trail_id, batch_seq_seed
        )

        ############################
        # S_i work arround. Es feo pero por ahora vale. 
        if config["model"]["name"] == "ZerlautAdaptationFirstOrder" and "S_i" in parameters:
            for i, (name, signal) in enumerate(stimulus):
                if name == "external_input_in_ex":
                    stimulus[i] =(name, signal * parameters["S_i"])
        ############################

        # 3. Simulation
        print(f"Simulating trial {simulation_id} (batch {batch_id}, trial {trail_id})...", flush=True) # LOG
        time_ms, states = simulate_trial(config, parameters, stimulus, len(time_ms))

        # 4. Almacenamiento de los resultados
        metadata = {
            "simulation_id": simulation_id, "batch_id": batch_id, "trial_id": trail_id,
            "parameters": parameters, "probability": float(np.prod(list(probabilities.values()))), 
            "probabilities": restore_flat_params(probabilities),
            **stimulus_metadata,
        }

        save_single_region_simulation_(file_path, time_ms, states, metadata)


def run_batch(config, batch_id, sobol_seed, batch_seq_seed, output_dir):
    started = perf_counter()
    output = StringIO()
    error = None

    # Salida de logging de TVB
    console = next(h for h in logging.getLogger("tvb.simulator").handlers 
                   if isinstance(h, logging.StreamHandler) and not isinstance(h, logging.FileHandler))
    tvb_out = console.stream
    console.setStream(output)    

    with redirect_stdout(output), redirect_stderr(output):
        try:
            simulate_batch(config, batch_id, sobol_seed, batch_seq_seed, Path(output_dir))
        except Exception as failure:
            error = failure
        finally:
            console.setStream(tvb_out)

    return {
        "batch_id": batch_id,
        "duration_s": perf_counter() - started,
        "logs": output.getvalue().splitlines(),
        "error": error,
    }

def run_batches(config, output_dir, *, first_batch=0, n_batches=None, n_workers=1):
    # Validación de la configuración
    validate_config(config)
    save_config(config, output_dir := Path(output_dir))

    # Finished batch log
    (completed_log_path := output_dir / "completed_batches.log").touch(exist_ok=True)
    completed = set(map(int, completed_log_path.read_text(encoding="utf-8").split()))

    # IO LOG
    (io_log_path := output_dir / "io.log").touch(exist_ok=True)

    # Seeds for Sobol sampling and batch sequence generation
    seed = config.get("seed", 42) #pop
    seeds = tuple(map( #sobol_seed, batch_seq_seed
        int, np.random.SeedSequence(seed).generate_state(2, dtype=np.uint64)
    ))

    # LOOP
    count, mean_duration_s = 0, 0.0
    
    with (
            ProcessPoolExecutor(max_workers=n_workers) as pool, 
            completed_log_path.open("a", buffering=1, encoding="utf-8") as clog,
            io_log_path.open("a", buffering=1, encoding="utf-8") as iolog
        ):
       
        futures = {
            pool.submit(run_batch, config, batch, *seeds, output_dir)
            for batch in range(first_batch, first_batch + (1 if n_batches is None else n_batches))
            if batch not in completed  # ToDo: Mejorar el reanuadar.
        }
       
        for future in as_completed(futures):
            futures.remove(future) # TODO:Libera memoria, pero puede que no merezca la
            batch = future.result()
            iolog.write(f'Batch {batch["batch_id"]}: {batch["duration_s"]:.2f} s\n')
            iolog.write('\n'.join(batch["logs"]) + '\n\n\n')
            if batch["error"] is not None: raise batch["error"]
            
            clog.write(f'{batch["batch_id"]}\n')  
            
            mean_duration_s += (batch["duration_s"] - mean_duration_s) / (count:=count+1)
            pprint(f'Batch {batch["batch_id"]}: {batch["duration_s"]:.2f} s; mean: {mean_duration_s:.2f} s')

    # Mejor alternativa para trabajar con numpy.     
    # result = Parallel(n_jobs=n_workers)( 
    #     delayed(run_batch)(config, batch, sobol_seed, batch_seq_seed)
    #     for batch in range(first_batch, first_batch + (1 if n_batches is None else n_batches))
    #     if batch not in completed
    # )

In [ ]:
run_batches(config, WORK_DIR / "results" / "experiment_01", 
            first_batch=0, n_batches=100, n_workers=34)

## Load and plot a simulation

In [ ]:
OUTPUT_DIR = WORK_DIR / "results" / "exp1"
TRIAL_IDS = [0,1,2,3,4,5,6,7,8,9,10,12,13,14,15] #201
CONFIG_PATH = OUTPUT_DIR / "config.json"

# PLOT_STATES = None
PLOT_STATES = ["E", "I"]

with CONFIG_PATH.open(encoding="utf-8") as config_file:
    loaded_config = json.load(config_file)

In [ ]:
from IPython.display import clear_output, display

n_rows = (len(TRIAL_IDS) + 3) // 4
fig, axes = plt.subplots(
    n_rows, 4, figsize=(24, 6 * max(n_rows, 4)), squeeze=False, constrained_layout=True,
)

plt.close(fig)
for ax in axes.flat[len(TRIAL_IDS):]:
    ax.set_axis_off()

for simulation_id, ax in zip(TRIAL_IDS, axes.flat):
    OUTPUT_PATH = OUTPUT_DIR / f"trial_{simulation_id:08d}.npz"

    with np.load(OUTPUT_PATH, allow_pickle=False) as saved:
        time_ms = np.asarray(saved["time_ms"])
        states = {name: np.asarray(saved[name]) for name in saved.files
                  if name not in ("time_ms", "metadata_json")}
        trial_metadata = json.loads(saved["metadata_json"].item())

    metadata = ResultMetadata(
        simulation=SimulationMetadata(
            **loaded_config["simulation"], output_dtype=loaded_config["output"]["dtype"],
        ),
        model=ModelMetadata(
            name=loaded_config["model"]["name"],
            parameters=trial_metadata["parameters"],
            output_variables=list(states),
            state_units={
                "E": "kHz", "I": "kHz",
                "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
                "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
            },
        ),
        stimulus=trial_metadata["stimulus"],
        stimulus_seed=trial_metadata["stimulus_seed"],
    )

    time_ms = time_ms[::4]
    states = {key: value[::4] for key, value in states.items()}
    fig, trial_axes = plot_results(
        time_ms, states, metadata,
        show_stimulus=True, ax=ax,
        plot_states=PLOT_STATES,
    )
    trial_axes[0].set_title(f"Trial {simulation_id}")
    clear_output(wait=True)
    display(fig)